# Unified game delisting statistics

This notebook analyzes the unified game dataset built from:

- **Delisted Games**
- **Steam Tracker**
- **Dead Game List** (Stop Killing Games wiki)
- **DoesItPlay**

## Main analyses

1. **Time evolution** of removal/delisting-like events by year, compared with IGDB release volume.
2. The same yearly analysis restricted to games that **overlap with DoesItPlay**.
3. Additional descriptive statistics for DoesItPlay offline-play and download-dependency information.

### Event definition

The unified dataset already contains the field `is_historically_unavailable`.  
This notebook treats rows where that field is `True` as a comparable **historical event-like subset**:

- Delisted Games → historically delisted records.
- Steam Tracker → delisted / purchase-disabled / unavailable-like records according to the aggregation pipeline.
- Dead Game List → dead/preserved/resurrected historical cases.
- DoesItPlay → **not used as an outcome source**, because it supplies playability/dependency observations rather than delisting labels.

`At Risk` records are therefore not counted as historical removals.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

# Paths are relative to this notebook's directory.
DATA_FILE = Path("../../Analysis/unified_games.csv")
IGDB_RELEASES_FILE = Path("../../IGDB_miner/igdb_games_released_by_year.csv")

OUTPUT_DIR = Path(".")

CURRENT_YEAR = pd.Timestamp.now().year

# Last year fully covered by the committed snapshots (mined during 2026).
# Later years are partial and are left out of every yearly series.
LAST_COMPLETE_YEAR = 2025

print("Data file:", DATA_FILE)
print("Output directory:", OUTPUT_DIR.resolve())
print("Last complete year in the yearly series:", LAST_COMPLETE_YEAR)


In [ ]:
# ---------------------------------------------------------------------
# Figure style
# ---------------------------------------------------------------------
# Shared with the other RQ3 analyses: see ../figure_style.py.

import sys
import textwrap

sys.path.insert(0, str(Path("..").resolve()))

from figure_style import (
    IGDB_STYLE,
    SOURCE_STYLE,
    TOTAL_STYLE,
    apply_style,
    plot_bars,
    plot_source_lines,
    set_title,
    source_label,
    style_count_axis,
    style_year_axis,
    wide_figure,
)

apply_style()

In [ ]:
# Load the unified dataset

df = pd.read_csv(DATA_FILE, low_memory=False)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")
print()
print("Rows by source:")
display(
    df["source"]
    .value_counts()
    .rename_axis("source")
    .reset_index(name="rows")
)


In [ ]:
# ---------------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------------

def coerce_nullable_bool(series: pd.Series) -> pd.Series:
    """Convert common CSV boolean representations into pandas BooleanDtype."""
    mapping = {
        True: True,
        False: False,
        "True": True,
        "False": False,
        "true": True,
        "false": False,
        "1": True,
        "0": False,
        1: True,
        0: False,
    }

    return series.map(mapping).astype("boolean")


def save_current_figure(filename: str):
    """Save the active figure as PNG and as vector PDF."""
    path = OUTPUT_DIR / filename
    plt.savefig(path)
    plt.savefig(path.with_suffix(".pdf"))
    print("Saved:", path)


def count_unique_titles_by_source(data: pd.DataFrame) -> pd.DataFrame:
    """Count unique normalized titles for each source."""
    return (
        data.dropna(subset=["normalized_title"])
        .groupby("source")["normalized_title"]
        .nunique()
        .sort_values(ascending=False)
        .rename("unique_titles")
        .reset_index()
    )

In [ ]:
# ---------------------------------------------------------------------
# Prepare canonical analysis fields
# ---------------------------------------------------------------------

bool_columns = [
    "is_explicitly_delisted",
    "is_historically_unavailable",
    "is_dead",
    "is_at_risk",
    "offline_play_bool",
    "download_required_bool",
]

for column in bool_columns:
    if column in df.columns:
        df[column] = coerce_nullable_bool(df[column])

df["event_year"] = pd.to_numeric(
    df["event_year"],
    errors="coerce",
).astype("Int64")

# Historical event-like records.
event_df = df[
    df["is_historically_unavailable"].fillna(False)
].copy()

# DoesItPlay is a reference/enrichment source.
doesitplay_df = df[
    df["source"].eq("doesitplay")
].copy()

doesitplay_titles = set(
    doesitplay_df["normalized_title"]
    .dropna()
    .loc[lambda s: s.ne("")]
)

event_df["overlaps_doesitplay"] = (
    event_df["normalized_title"].isin(doesitplay_titles)
)

overlap_event_df = event_df[
    event_df["overlaps_doesitplay"]
].copy()

# Separate complete historical years from partial and future years.
historical_event_df = event_df[
    event_df["event_year"].notna()
    & event_df["event_year"].le(LAST_COMPLETE_YEAR)
].copy()

future_event_df = event_df[
    event_df["event_year"].notna()
    & event_df["event_year"].gt(CURRENT_YEAR)
].copy()

historical_overlap_event_df = overlap_event_df[
    overlap_event_df["event_year"].notna()
    & overlap_event_df["event_year"].le(LAST_COMPLETE_YEAR)
].copy()

# Every yearly figure shares this range so their year axes line up.
YEAR_RANGE = (
    int(historical_event_df["event_year"].min()),
    LAST_COMPLETE_YEAR,
)

print(f"Historical event-like rows: {len(event_df):,}")
print(
    "Unique historical event-like titles:",
    f"{event_df['normalized_title'].nunique():,}",
)
print(
    "Unique DoesItPlay titles:",
    f"{doesitplay_df['normalized_title'].nunique():,}",
)
print(
    "Event-like titles overlapping DoesItPlay:",
    f"{overlap_event_df['normalized_title'].nunique():,}",
)


## 1. Dataset overview

The first plot shows the number of **unique normalized game titles** represented by each source.  
This is preferable to raw row counts because DoesItPlay may contain multiple tests for the same game across platform/version/hardware combinations.


In [ ]:
source_title_counts = count_unique_titles_by_source(df)

display(source_title_counts)

plot_bars(
    [source_label(s) for s in source_title_counts["source"]],
    source_title_counts["unique_titles"],
    [SOURCE_STYLE[s]["color"] for s in source_title_counts["source"]],
    title="Unique game titles by source",
    value_label="Unique normalized titles",
)

save_current_figure("01_unique_titles_by_source.png")
plt.show()

## 2. Historical event-like games by source

This plot counts unique titles marked as historically unavailable/dead/delisted-like by the unified pipeline.

DoesItPlay does not appear here because it is not used as an outcome source.


In [ ]:
event_source_counts = count_unique_titles_by_source(event_df)

display(event_source_counts)

plot_bars(
    [source_label(s) for s in event_source_counts["source"]],
    event_source_counts["unique_titles"],
    [SOURCE_STYLE[s]["color"] for s in event_source_counts["source"]],
    title="Historical delisting / unavailability-like titles by source",
    value_label="Unique normalized titles",
)

save_current_figure("02_event_titles_by_source.png")
plt.show()

# 3. Time evolution of removals

The yearly series counts **unique normalized titles within each source/year**.

Only complete years (up to `LAST_COMPLETE_YEAR`) are plotted; the partial current year is left out.  
Future announced event years are reported separately below.


In [ ]:
# ---------------------------------------------------------------------
# Plot 2 requested:
# Time evolution of event-like removals per year
# ---------------------------------------------------------------------

yearly_by_source = (
    historical_event_df
    .dropna(subset=["normalized_title", "event_year"])
    .groupby(["event_year", "source"])["normalized_title"]
    .nunique()
    .rename("unique_event_titles")
    .reset_index()
)

yearly_pivot = (
    yearly_by_source
    .pivot(
        index="event_year",
        columns="source",
        values="unique_event_titles",
    )
    .fillna(0)
    .sort_index()
)

display(yearly_pivot.tail(15))

plot_source_lines(
    yearly_pivot,
    title="Historical delisting / unavailability-like events by year",
    xlabel="Event year",
    ylabel="Unique normalized titles",
    year_range=YEAR_RANGE,
)

save_current_figure("04_event_evolution_by_year_and_source.png")
plt.show()

yearly_by_source.to_csv(
    OUTPUT_DIR / "event_counts_by_year_and_source.csv",
    index=False,
)

In [ ]:
# Total unique event-like titles by year across all outcome sources.
#
# A title appearing in more than one source in the same year is counted once.
# This cell also integrates the IGDB yearly release counts and evaluates
# whether yearly release volume is statistically associated with the number
# of historical delisting / unavailability-like events.

from scipy.stats import pearsonr, spearmanr

yearly_total = (
    historical_event_df
    .dropna(subset=["normalized_title", "event_year"])
    .groupby("event_year")["normalized_title"]
    .nunique()
    .rename("unique_event_titles")
    .reset_index()
)

yearly_total["event_year"] = pd.to_numeric(
    yearly_total["event_year"],
    errors="coerce",
).astype("Int64")

# ---------------------------------------------------------------------
# Load IGDB release-volume denominator
# ---------------------------------------------------------------------

igdb_releases = pd.read_csv(IGDB_RELEASES_FILE)

required_igdb_columns = {"year", "igdb_games_released"}
missing_igdb_columns = required_igdb_columns - set(igdb_releases.columns)

if missing_igdb_columns:
    raise ValueError(
        "IGDB yearly-release file is missing required columns: "
        + ", ".join(sorted(missing_igdb_columns))
    )

igdb_releases["year"] = pd.to_numeric(
    igdb_releases["year"],
    errors="coerce",
).astype("Int64")

igdb_releases["igdb_games_released"] = pd.to_numeric(
    igdb_releases["igdb_games_released"],
    errors="coerce",
)

# Keep only years present in both sources.
yearly_with_igdb = (
    yearly_total
    .merge(
        igdb_releases,
        left_on="event_year",
        right_on="year",
        how="inner",
        validate="one_to_one",
    )
    .drop(columns="year")
    .dropna(subset=["unique_event_titles", "igdb_games_released"])
    .sort_values("event_year")
    .reset_index(drop=True)
)

display(yearly_with_igdb.tail(15))

# ---------------------------------------------------------------------
# Combined historical plot
# ---------------------------------------------------------------------
# Counts have very different magnitudes, therefore two y-axes are used.

fig, ax_delisted = wide_figure()

plot_years = yearly_with_igdb["event_year"].astype(int)

ax_released = ax_delisted.twinx()

line_released = ax_released.plot(
    plot_years,
    yearly_with_igdb["igdb_games_released"],
    **IGDB_STYLE,
    markeredgecolor="white",
    markeredgewidth=0.4,
    label="IGDB games released",
    clip_on=False,
)

line_delisted = ax_delisted.plot(
    plot_years,
    yearly_with_igdb["unique_event_titles"],
    **TOTAL_STYLE,
    markeredgecolor="white",
    markeredgewidth=0.4,
    label="Historical delisting / unavailability-like titles",
    clip_on=False,
)

# Draw the event-like series above the IGDB series.
ax_delisted.set_zorder(ax_released.get_zorder() + 1)
ax_delisted.patch.set_visible(False)

ax_delisted.set_xlabel("Year")
ax_delisted.set_ylabel("Unique historical event-like titles")
style_year_axis(ax_delisted, YEAR_RANGE)
style_count_axis(ax_delisted)

ax_released.set_ylabel("IGDB games released")
ax_released.grid(False)
ax_released.spines["right"].set_visible(True)
style_count_axis(ax_released)

lines = line_delisted + line_released
labels = [line.get_label() for line in lines]
ax_delisted.legend(lines, labels, loc="upper left")

set_title(
    ax_delisted,
    "Historical event-like titles and IGDB game releases by year",
)

save_current_figure(
    "05_event_evolution_total_with_igdb_releases.png"
)
plt.show()

# ---------------------------------------------------------------------
# Correlation analysis
# ---------------------------------------------------------------------
# Raw time-series levels can show a large correlation simply because both
# variables trend over time. Therefore we report:
#
# 1. Pearson correlation on annual levels: linear association.
# 2. Spearman correlation on annual levels: monotonic association.
# 3. Pearson correlation on first differences: linear association between
#    year-to-year changes after removing the common level/trend component.
# 4. Spearman correlation on first differences: monotonic association between
#    year-to-year changes.
#
# Only complete years (up to LAST_COMPLETE_YEAR) enter the tests, as in the
# plot above.

correlation_df = yearly_with_igdb[
    yearly_with_igdb["event_year"] <= LAST_COMPLETE_YEAR
].copy()

if len(correlation_df) < 4:
    raise ValueError(
        "Too few complete overlapping years for correlation analysis."
    )

x = correlation_df["igdb_games_released"].astype(float)
y = correlation_df["unique_event_titles"].astype(float)

pearson_levels = pearsonr(x, y)
spearman_levels = spearmanr(x, y)

# First differences: year-to-year changes.
diff_df = correlation_df[
    ["event_year", "igdb_games_released", "unique_event_titles"]
].copy()

diff_df["delta_igdb_games_released"] = (
    diff_df["igdb_games_released"].diff()
)

diff_df["delta_event_titles"] = (
    diff_df["unique_event_titles"].diff()
)

diff_df = diff_df.dropna(
    subset=["delta_igdb_games_released", "delta_event_titles"]
)

pearson_diff = pearsonr(
    diff_df["delta_igdb_games_released"],
    diff_df["delta_event_titles"],
)

spearman_diff = spearmanr(
    diff_df["delta_igdb_games_released"],
    diff_df["delta_event_titles"],
)

ALPHA = 0.05

correlation_results = pd.DataFrame(
    [
        {
            "series": "annual levels",
            "test": "Pearson",
            "correlation": pearson_levels.statistic,
            "p_value": pearson_levels.pvalue,
            "n": len(correlation_df),
        },
        {
            "series": "annual levels",
            "test": "Spearman",
            "correlation": spearman_levels.statistic,
            "p_value": spearman_levels.pvalue,
            "n": len(correlation_df),
        },
        {
            "series": "first differences",
            "test": "Pearson",
            "correlation": pearson_diff.statistic,
            "p_value": pearson_diff.pvalue,
            "n": len(diff_df),
        },
        {
            "series": "first differences",
            "test": "Spearman",
            "correlation": spearman_diff.statistic,
            "p_value": spearman_diff.pvalue,
            "n": len(diff_df),
        },
    ]
)

correlation_results["significant_at_0.05"] = (
    correlation_results["p_value"] < ALPHA
)

correlation_results["correlation"] = (
    correlation_results["correlation"].round(4)
)
correlation_results["p_value"] = (
    correlation_results["p_value"].map(lambda value: f"{value:.4g}")
)

print(
    f"Correlation tests use complete years "
    f"{int(correlation_df['event_year'].min())}-"
    f"{int(correlation_df['event_year'].max())}; "
    f"partial years after {LAST_COMPLETE_YEAR} are excluded."
)

display(correlation_results)

# Interpretation focused on the de-trended / first-difference result.
pearson_diff_significant = pearson_diff.pvalue < ALPHA
spearman_diff_significant = spearman_diff.pvalue < ALPHA

if pearson_diff_significant or spearman_diff_significant:
    print(
        "At least one first-difference test is statistically significant "
        f"at alpha={ALPHA}. This provides evidence that year-to-year "
        "changes in IGDB release volume and event-like counts co-vary."
    )
else:
    print(
        "The first-difference correlations are not statistically significant "
        f"at alpha={ALPHA}. A significant correlation in the raw annual "
        "levels should therefore be interpreted cautiously: it is consistent "
        "with a shared long-term time trend and does not by itself demonstrate "
        "that yearly changes in release volume are associated with yearly "
        "changes in delisting / unavailability-like events."
    )

# ---------------------------------------------------------------------
# Exports
# ---------------------------------------------------------------------

yearly_total.to_csv(
    OUTPUT_DIR / "event_counts_by_year_total.csv",
    index=False,
)

yearly_with_igdb.to_csv(
    OUTPUT_DIR / "event_counts_and_igdb_releases_by_year.csv",
    index=False,
)

correlation_results.to_csv(
    OUTPUT_DIR / "event_release_correlation_tests.csv",
    index=False,
)

In [ ]:
# Check for event dates later than the current calendar year.

if future_event_df.empty:
    print("No future event years are present.")
else:
    future_summary = (
        future_event_df[
            [
                "source",
                "game_name",
                "normalized_title",
                "event_year",
                "source_status",
            ]
        ]
        .drop_duplicates()
        .sort_values(["event_year", "source", "game_name"])
    )

    print(
        "These rows have future event years and are excluded from "
        "historical-removal plots:"
    )
    display(future_summary)

    future_summary.to_csv(
        OUTPUT_DIR / "future_announced_events.csv",
        index=False,
    )


# 4. Games overlapping DoesItPlay

An overlap is defined by an exact match on the existing `normalized_title`.

This is useful for combining a historical outcome-like record with DoesItPlay's physical/offline-play observations, but title matching is still only an exploratory entity-resolution method.


In [ ]:
# ---------------------------------------------------------------------
# Overall overlap statistics
# ---------------------------------------------------------------------

overlap_summary = (
    event_df
    .groupby("source")
    .agg(
        event_titles=("normalized_title", "nunique"),
        doesitplay_overlap_titles=(
            "overlaps_doesitplay",
            lambda x: event_df.loc[x.index]
            .loc[x, "normalized_title"]
            .nunique()
        ),
    )
    .reset_index()
)

overlap_summary["overlap_percent"] = (
    100
    * overlap_summary["doesitplay_overlap_titles"]
    / overlap_summary["event_titles"]
)

display(overlap_summary)

plot_bars(
    [source_label(s) for s in overlap_summary["source"]],
    overlap_summary["doesitplay_overlap_titles"],
    [SOURCE_STYLE[s]["color"] for s in overlap_summary["source"]],
    title="Historical event-like titles also represented in DoesItPlay",
    value_label="Unique overlapping titles",
)

save_current_figure("06_doesitplay_overlap_by_source.png")
plt.show()

overlap_summary.to_csv(
    OUTPUT_DIR / "doesitplay_overlap_summary.csv",
    index=False,
)

## 4.1 Time evolution for event-like games overlapping DoesItPlay

Only complete years (up to `LAST_COMPLETE_YEAR`) are included.


In [ ]:
# ---------------------------------------------------------------------
# Requested overlap yearly evolution
# ---------------------------------------------------------------------

overlap_yearly_by_source = (
    historical_overlap_event_df
    .dropna(subset=["normalized_title", "event_year"])
    .groupby(["event_year", "source"])["normalized_title"]
    .nunique()
    .rename("unique_overlap_event_titles")
    .reset_index()
)

overlap_yearly_pivot = (
    overlap_yearly_by_source
    .pivot(
        index="event_year",
        columns="source",
        values="unique_overlap_event_titles",
    )
    .fillna(0)
    .sort_index()
)

display(overlap_yearly_pivot.tail(15))

plot_source_lines(
    overlap_yearly_pivot,
    title="Historical events by year for games also represented in DoesItPlay",
    xlabel="Event year",
    ylabel="Unique overlapping titles",
    year_range=YEAR_RANGE,
)

save_current_figure(
    "08_doesitplay_overlap_event_evolution.png"
)
plt.show()

overlap_yearly_by_source.to_csv(
    OUTPUT_DIR / "doesitplay_overlap_event_counts_by_year.csv",
    index=False,
)

# 5. DoesItPlay dependency characteristics for overlap games

DoesItPlay can contain several observations for one title.

For each title, the notebook derives:

- number of DoesItPlay test rows,
- whether **any** test says offline play is unavailable,
- whether **any** test says a download is required.

These are descriptive indicators only; they should not yet be interpreted as causal predictors of delisting.


In [ ]:
# ---------------------------------------------------------------------
# Aggregate DoesItPlay observations per normalized title
# ---------------------------------------------------------------------

dip = doesitplay_df.copy()

dip["offline_play_bool"] = coerce_nullable_bool(
    dip["offline_play_bool"]
)

dip["download_required_bool"] = coerce_nullable_bool(
    dip["download_required_bool"]
)

dip_title_features = (
    dip.dropna(subset=["normalized_title"])
    .groupby("normalized_title")
    .agg(
        doesitplay_test_rows=("normalized_title", "size"),
        any_offline_unavailable=(
            "offline_play_bool",
            lambda s: bool((s == False).any()),
        ),
        any_download_required=(
            "download_required_bool",
            lambda s: bool((s == True).any()),
        ),
    )
    .reset_index()
)

overlap_titles = (
    overlap_event_df[
        ["normalized_title", "game_name"]
    ]
    .drop_duplicates(subset=["normalized_title"])
)

overlap_dependency = overlap_titles.merge(
    dip_title_features,
    on="normalized_title",
    how="left",
)

display(overlap_dependency.head(20))

dependency_counts = pd.DataFrame(
    {
        "indicator": [
            "At least one test cannot play offline",
            "At least one test requires download",
        ],
        "titles": [
            int(overlap_dependency["any_offline_unavailable"].sum()),
            int(overlap_dependency["any_download_required"].sum()),
        ],
    }
)

display(dependency_counts)

plot_bars(
    [textwrap.fill(i, 22) for i in dependency_counts["indicator"]],
    dependency_counts["titles"],
    SOURCE_STYLE["doesitplay"]["color"],
    title="DoesItPlay dependency indicators among overlapping event-like games",
    value_label="Unique normalized titles",
)

save_current_figure(
    "09_doesitplay_overlap_dependency_indicators.png"
)
plt.show()

overlap_dependency.to_csv(
    OUTPUT_DIR / "doesitplay_overlap_dependency_features.csv",
    index=False,
)

# 6. Export compact analysis tables

The notebook already saves plot-specific tables as it runs.  
This final cell exports the main analysis subsets for downstream inspection.


In [ ]:
event_export_columns = [
    "source",
    "source_record_id",
    "game_name",
    "normalized_title",
    "publisher",
    "developer",
    "platforms",
    "event_info",
    "event_year",
    "source_status",
    "is_historically_unavailable",
    "overlaps_doesitplay",
]

event_df[
    [c for c in event_export_columns if c in event_df.columns]
].to_csv(
    OUTPUT_DIR / "historical_event_like_records.csv",
    index=False,
)

overlap_event_df[
    [c for c in event_export_columns if c in overlap_event_df.columns]
].to_csv(
    OUTPUT_DIR / "historical_event_like_records_overlapping_doesitplay.csv",
    index=False,
)

print("Analysis completed.")
print("Outputs written to:", OUTPUT_DIR.resolve())
